In [1]:
# Sam Brown
# sam_brown@mines.edu
# June19
# Goal: Preprocess data and create dataframe for analysis of long-term slip patterns

# Get tide data for entire timeframe using average gz coords (no tide in la stations but timing is still relevant)

import sys
sys.path.append("../")

import my_lib.funcs
import stations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# NOTES: use GZ05 for coordinates to retrieve tide data.

# For this dataset we are focusing on tidal modulation. For each event we will need tide height, tide derivative, form factor, 
# time since last event, slip size (standardized for each station and averaged), high or low tide event, and some indicator
# to signal whether the event is following a skipped low/high tide event.

In [3]:
# Load the paths

# df_2008 = my_lib.funcs.load_evt("/Users/sambrown/Documents/SURF/Events/2008_2008Events2stas")
# df_2009 = my_lib.funcs.load_evt("/Users/sambrown/Documents/SURF/Events/2009_2009Events2stas")
df_2010 = my_lib.funcs.load_evt("/Users/sambrown/Documents/SURF/Events/2010_2010Events2stas")
df_2011 = my_lib.funcs.load_evt("/Users/sambrown/Documents/SURF/Events/2011_2011Events2stas")
df_2012 = my_lib.funcs.load_evt("/Users/sambrown/Documents/SURF/Events/2012_2012Events2stas")
df_2013 = my_lib.funcs.load_evt("/Users/sambrown/Documents/SURF/Events/2013_2013Events2stas")
df_2014 = my_lib.funcs.load_evt("/Users/sambrown/Documents/SURF/Events/2014_2014Events2stas")
df_2015 = my_lib.funcs.load_evt("/Users/sambrown/Documents/SURF/Events/2015_2015Events2stas")
df_2016 = my_lib.funcs.load_evt("/Users/sambrown/Documents/SURF/Events/2016_2016Events2stas")
df_2017 = my_lib.funcs.load_evt("/Users/sambrown/Documents/SURF/Events/2017_2017Events2stas")
df_2018 = my_lib.funcs.load_evt("/Users/sambrown/Documents/SURF/Events/2018_2018Events2stas")
df_2019 = my_lib.funcs.load_evt("/Users/sambrown/Documents/SURF/Events/2019_2019Events2stas")

# One Large list
all_dfs = (
    df_2010 + df_2011 + df_2012 + df_2013 +
    df_2014 + df_2015 + df_2016 + df_2017 + df_2018 + df_2019
)


In [4]:
# Preprocess
clean_df = my_lib.funcs.extract_event_features(all_dfs)

In [ ]:
clean_df[5].head()

In [5]:
standards = pd.read_csv('../station_standards.csv')

In [ ]:
standards.head(10)

In [6]:
# Loop through each event. Loop through each station, append standardized delta in list then average it.
# add start time to data frame

# Initialize dataframe
net_df = pd.DataFrame(columns = ['tide_h', 'tide_deriv', 'form_fac', 'time_since', 'slip_size_standardized', 'high_t_evt', 'start_time'])

for event in clean_df:

    # Initialize list for slip sizes
    slip_deltas = []

    # Loop through rows
    for i, row in event.iterrows():
        station = row['station'][:4]
        if station == 'slw1' or station =='ws04' or station =='ws05':
            continue

        row_sch = standards[standards['Station'] == station]

        # if row_sch.empty:
        #     raise ValueError(f"Station not found in standards: '{station}'")

        # Min Max scaling
        

        
        # Standardization metrics
        station_mean = row_sch['slip_size'].values
        station_sd = row_sch['slip_size_sd'].values

        # print(f"{row['total_delta']}, {station}, {station_mean}, {station_sd}")
        
        standardized_val = (row['total_delta'] - station_mean) / station_sd

        slip_deltas.append(standardized_val)

    net_df.loc[len(net_df)] = {
        "slip_size_standardized": sum(slip_deltas) / len(slip_deltas),
        "start_time": event.at[0, 'start_time']
    }

In [7]:
# Load tide Data

#Average coordinates for gz stations (source code in severity_class nb)
x_cor = -168955.1491394913 
y_cor = -599694.5432784811

tide_df = my_lib.funcs.get_tide_height(4380, x_cor, y_cor, "2008-01-01 00:00:00") # 12 years worth of data

Elapsed time: 66.54851007461548 seconds


In [8]:
# Organize events by time
net_df = net_df.sort_values('start_time')

In [9]:
# Calculate time since in minutes
net_df['time_since'] = net_df['start_time'].diff().dt.total_seconds() / 60

In [10]:
# Need to get start times down to minutes
tide_df['time'] = tide_df['time'].apply(lambda x: x.strftime("%Y-%m-%d %H:%M"))
net_df['start_time'] = net_df['start_time'].apply(lambda x: x.strftime("%Y-%m-%d %H:%M"))

In [11]:
# Insert Tide values
net_df['start_time'] = pd.to_datetime(net_df['start_time'])
tide_df['time'] = pd.to_datetime(tide_df['time'])

# Merge tide height into net_df based on matching timestamps
merged_df = pd.merge(net_df, tide_df[['time', 'tide_height']], 
                     left_on='start_time', right_on='time', how='left')

# Drop extra 'time' column if you want
merged_df = merged_df.drop(columns=['time'])

In [12]:
# Insert tide derivatives into data
tide_d = my_lib.funcs.tide_derivative(tide_df)
for i, row in merged_df.iterrows():
    time = row['start_time']

    index = tide_d[tide_d['time'] == time].index

    if not index.empty:
        idx = index[0]
        merged_df.at[i, 'tide_deriv'] = tide_d.at[idx, 'tide_deriv']

In [15]:
merged_df.tail(40)

,tide_h,tide_deriv,form_fac,time_since,slip_size_standardized,high_t_evt,start_time,tide_height
4536,NaN,-0.154708,NaN,1515.00,[1.0002027564637628],NaN,2019-10-22 19:46:00,63.39968
4537,NaN,-0.109705,NaN,1463.00,[0.9817501167187329],NaN,2019-10-23 20:09:00,40.670672
4538,NaN,-0.169613,NaN,692.00,[-0.6151748926423002],NaN,2019-10-24 07:41:00,-57.010748
4539,NaN,-0.230658,NaN,1260.00,[0.39107396826281837],NaN,2019-10-25 04:41:00,11.443025
4540,NaN,-0.253696,NaN,845.00,[-0.39235750360154986],NaN,2019-10-25 18:46:00,10.004411
4541,NaN,-0.341208,NaN,740.00,[-0.8402042835694482],NaN,2019-10-26 07:06:00,-5.371641
4542,NaN,-0.369750,NaN,790.00,[-1.20937435911228],NaN,2019-10-26 20:16:00,-27.245628
4543,NaN,0.106211,NaN,908.25,[-1.8924984843717605],NaN,2019-10-27 11:24:00,-26.332719
4544,NaN,-0.223927,NaN,1316.75,[0.6327371970896399],NaN,2019-10-28 09:21:00,9.035257
4545,NaN,-0.561549,NaN,675.00,[-1.0113385684479346],NaN,2019-10-28 20:36:00,-23.040563


In [16]:
# Form factor calculation

form_fac = my_lib.funcs.form_factor_calc(tide_df)

/Users/sambrown/whillans-surf/notebooks/SURF/neural_nets/../my_lib/funcs.py:411: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, pcov = scipy.optimize.curve_fit(sines, seconds_tide, tide_window, p0=initial_guess)


In [17]:
# Add date-only column to form_fac
form_fac['date_only'] = form_fac['dates'].dt.date

# Loop through each row in avg_dat
for i, event in merged_df.iterrows():
    time = event['start_time']
    target_date = time.date()

    # Select all rows with matching date
    rows_date = form_fac[form_fac['date_only'] == target_date]

    # Compute average form factor for that date
    merged_df.at[i, 'form_fac'] = rows_date['form_factors'].mean()

In [18]:
# Encode high tide vs low tide event
merged_df['high_t_evt'] = (merged_df['tide_h'] > 0).astype(int)

In [19]:
merged_df.head()

,tide_h,tide_deriv,form_fac,time_since,slip_size_standardized,high_t_evt,start_time,tide_height
0,NaN,0.023978,3.777526,NaN,[-1.9313504867503009],0,2010-01-01 15:25:00,109.008588
1,NaN,-0.230950,3.777526,512.50,[-2.792975595983694],0,2010-01-01 23:58:00,-126.504019
2,NaN,-0.088248,3.121720,1037.75,[-0.7939203106706463],0,2010-01-02 17:15:00,94.293312
3,NaN,0.138839,2.567067,1382.75,[-0.8381986177674853],0,2010-01-03 16:18:00,65.146216
4,NaN,-0.484389,2.567067,445.00,[-1.4355941749658114],0,2010-01-03 23:43:00,-57.846854


In [100]:
merged_df['slip_size'] = merged_df['slip_size'].str[0]


In [ ]:
# We would like to add feature(s) that takes the time period between events and retrieves the form factor for the time period between these events. 
# Two different features, semi diurnal fit and diurnal fit 

# Loop through each event 

    # get time period between events

    # calculate form factor for this time period

    # insert the features into the data set for that row

# So the features for a given event will be the coefficinets for form factor calc leading up to this event.

In [106]:
# Export to csv file
merged_df.to_csv('09-18', index = False)